# Chapter 13: Digital Forensics

> "The goal of digital forensics is to recover and analyze evidence in a manner that preserves
> its integrity so that it can be used in a legal proceeding." standard forensic doctrine

---


## Learning Objectives

After completing this chapter, you will be able to:

1. Explain the principles of digital forensics and the chain of custody.
2. Describe the forensic acquisition process and the difference between logical and physical images.
3. Apply hash verification to prove evidence integrity.
4. Describe file system artefacts including deleted files, slack space, and timestamps.
5. Explain memory forensics and what can be recovered from a RAM dump.
6. Describe network forensics and the value of PCAP analysis.
7. Explain log analysis in a forensic context and the significance of timestamp correlation.
8. Recognize anti-forensic techniques and their countermeasures.


## Key Terms

- **Order of volatility**: collect most-ephemeral evidence first (registers->RAM->network->disk->backups).
- **Chain of custody**: documented unbroken record of evidence handling for admissibility.
- **Write blocker / forensic image / hashing**: tools and steps that preserve and prove evidence integrity.
- **Daubert standard / Federal Rules of Evidence**: legal tests for admissible expert/forensic methods.

- **AI evidence triage**: machine-learning prioritization of which artifacts to analyze first.
- **IoT forensics**: forensics of smart-device, network, and cloud evidence.
- **Explainable AI (XAI)**: interpretable model outputs needed for admissible AI-assisted forensics.
- **Cyberattack attribution detection (CAD)**: identifying an attack's source/characteristics from forensic traces.

- **Chain of custody**: documented record of who had custody of evidence and when.
- **Forensic image**: a bit-for-bit copy of a storage device including unallocated space.
- **Write blocker**: hardware or software preventing any write to the source device during acquisition.
- **Hash verification**: computing a cryptographic hash of the image to prove it is unchanged.
- **Deleted file recovery**: recovering files whose directory entries have been removed but data not overwritten.
- **Slack space**: unused space in a file's last allocated cluster, potentially containing old data.
- **MFT**: Master File Table; NTFS structure recording metadata for every file.
- **Memory forensics**: acquiring and analyzing the contents of RAM.
- **Volatility**: open-source memory forensics framework.
- **PCAP**: packet capture; a file recording raw network traffic.
- **Artefact**: forensically significant data recovered from a system.
- **Anti-forensics**: techniques used to destroy or hide digital evidence.

---


## 13.1 Forensic Principles

### The Locard Exchange Principle

Locard's exchange principle from physical forensics holds that every contact leaves a trace.
In digital environments: every user action creates artefacts. Login events appear in logs, web
browsing creates browser history and DNS cache entries, file access updates timestamps and
prefetch records, and even deleted files often leave traces in unallocated space, the MFT, or
log files. The forensic investigator's task is to find, preserve, and interpret these traces.

### Forensic Soundness

Forensic soundness requires that the investigative process does not alter the evidence being
examined. Working from a verified copy (forensic image) of the original device, using a write
blocker during acquisition, computing hashes before and after, and maintaining a chain of custody
record are the foundational practices that allow a court to trust the evidence presented.

#### Chain of Custody

The chain of custody is a chronological record of every person who had control of the evidence
and every transfer between custodians. A break in the chain (unrecorded transfer, missing
signature) may allow a defense to argue that evidence was tampered with. For corporate
investigations that may lead to litigation or criminal referral, forensic investigators must
follow the same standards as law enforcement.

---


<!--CH13PROC-->
### The Forensic Process, Order of Volatility, and Chain of Custody

The principles above become a disciplined process whose every step must withstand legal scrutiny. The widely
used phases (NIST SP 800-86 and the ACPO model) are **identification, preservation, collection/acquisition,
examination, analysis, and reporting**. Two rules govern collection. First, the **order of volatility**:
collect the most ephemeral evidence first, CPU registers and cache, then RAM and running state, then network
connections, then disk, then archival media and backups, because volatile data vanishes on power loss. Second,
**preserve the original**: acquire a bit-for-bit **forensic image** using a **write blocker** so the source is
never altered, and **hash** both source and image (Chapter 2) to prove they match, repeating the hash later to
prove the evidence is unchanged (tamper-evidence in action).

Underpinning everything is the **chain of custody**: a documented, unbroken record of who handled the evidence,
when, why, and how, from seizure to courtroom. A gap in that chain, or a missing hash, can render even
damning evidence inadmissible. The four ACPO principles capture the ethic: do not change the data; if you must
access original data, be competent and explain why; keep an audit trail that a third party could reproduce; and
the case lead is responsible for compliance. This rigor is exactly what separates forensics from ordinary
incident analysis (Chapter 14), and it is the reason the AI-assisted methods earlier in this chapter must
remain explainable and human-validated.


## 13.2 Evidence Acquisition

### Write Blockers

A write blocker is inserted between the source device and the forensic workstation to prevent
any accidental write. Hardware write blockers (Tableau, WiebeTech) are preferred in legal
investigations because they are transparent to the OS and cannot be subverted by software. A
software write blocker (registry key in Windows, `blockdev --setro` in Linux) is acceptable in
internal investigations but may not withstand legal scrutiny.

### Forensic Imaging

A forensic image is a bit-for-bit copy of the entire device including allocated files, deleted
files, unallocated space, and slack space. `dd`, `dcfldd` (with hashing), and FTK Imager are
common tools. The image is usually stored in E01 (Expert Witness Format) which includes built-in
hash verification and compression.

#### Logical Versus Physical Acquisition

A logical acquisition copies only the file system's visible files and folders: faster and smaller
but misses deleted files and unallocated space. A physical (bit-level) acquisition copies the
entire device: slower but preserves all recoverable artefacts. For live systems, a cloud snapshot
or VM snapshot is an alternative when full physical imaging is impractical.

### Hash Verification

After acquisition, a hash (SHA-256) of the image is computed and recorded. At any later point,
re-hashing the image confirms it has not been modified. If the image hash matches the original,
the evidence is provably unchanged. SHA-256 is preferred over MD5 (which is cryptographically
broken) for new investigations, though MD5 is still widely used in practice due to tool defaults.

---


## 13.3 File System Forensics

### NTFS Artefacts

#### The Master File Table

The NTFS MFT records metadata for every file and directory: filename, timestamps (created, modified,
accessed, MFT-changed), size, and a reference to the file's data clusters. When a file is deleted,
its MFT entry is marked as available but is not immediately overwritten. The entry may remain
intact for a long time on a lightly-used volume, allowing filename and timestamp recovery even when
the file data is gone.

#### Timestamps and Timestomping

NTFS maintains four timestamps per file (MACB: Modified, Accessed, Changed, Born). Timestamp
correlation between MFT, log files, and the $LogFile/$UsnJrnl change journal allows an investigator
to reconstruct a precise timeline. Anti-forensic tools can alter timestamps (timestomping) but
often fail to update all four NTFS timestamps consistently, leaving detectable anomalies.

### Deleted Files and Unallocated Space

When a file is deleted, its directory entry is marked as available and its clusters are returned
to the free-space bitmap, but the data is not overwritten. Data recovery tools (Autopsy, FTK,
PhotoRec) scan unallocated clusters for known file signatures (magic bytes) to carve deleted files.
Solid-state drives with TRIM enabled may zero deleted blocks immediately, reducing recovery prospects.

---


### Disk, Memory, and Mobile Forensics in Practice

With the process fixed, the practical craft spans several evidence types. **Disk and file-system forensics**
recovers far more than visible files: a forensic image is searched for **deleted files** (whose directory entry
is removed but whose data persists until overwritten), **file carving** reconstructs files from raw bytes by
header/footer signatures, and **slack space** and unallocated areas hide data. Investigators build a
**timeline** from filesystem timestamps (the NTFS Master File Table, the `$MFT`) and from the Windows
**registry**, browser history, and event logs to reconstruct user activity. Tools include `dd` and **FTK
Imager** for imaging, **Autopsy/The Sleuth Kit** for analysis, and `bulk_extractor` for carving.

**Memory forensics** analyzes a RAM capture with frameworks such as **Volatility** to recover running
processes, network connections, injected code, and encryption keys, evidence that never touches disk and is the
only way to catch fileless malware (Chapter 15). **Mobile and cloud forensics** extend the discipline to phones
(logical and physical extractions, often via tools like Cellebrite) and to data held by cloud providers (legal
process, API acquisition), which is increasingly where evidence lives. A short worked example ties it together.

```{admonition} Worked Example: verifying and timelining an image
:class: note
After seizing a laptop, an examiner (1) attaches the drive through a hardware **write blocker**; (2) images it
with `dd` or FTK Imager and records `sha256sum` of source and image, confirming they match; (3) loads the image
in Autopsy, recovers deleted files by carving, and exports the `$MFT`; (4) builds a **super-timeline** (e.g.,
with Plaso/log2timeline) merging filesystem, registry, and log timestamps; (5) writes findings with each
artifact's hash and the chain-of-custody record. If the laptop was still powered on at seizure, a RAM capture
(for Volatility) would be taken *first*, per the order of volatility.
```


## 13.4 Memory Forensics

### Why Memory Matters

RAM contains information that is never written to disk: decrypted encryption keys, running process
lists, network connections, command history, clipboard contents, and injected shellcode. An attacker
using fileless malware may leave almost no disk artefacts while leaving extensive memory artefacts.
Memory acquisition must occur before the system is powered off; shutdown destroys volatile evidence.

### Acquiring Memory

On Windows, tools including Magnet RAM Capture and WinPmem produce a memory dump. On Linux,
`/dev/mem` (if accessible), `LiME` (Loadable Kernel Module), and hypervisor snapshots provide
memory. Live acquisition from a running system is preferable to cold-boot attacks (briefly cooling
DRAM to slow decay and transplanting to another machine) which are physically invasive.

### Analyzing Memory with Volatility

The Volatility framework analyses memory dumps across Windows, Linux, and macOS. Key plugins:

| Plugin | Information extracted |
|---|---|
| `pslist / pstree` | Running processes and their hierarchy |
| `dlllist` | DLLs loaded into a process |
| `netscan` | Network connections and listening sockets |
| `cmdline` | Command-line arguments for each process |
| `malfind` | Memory regions with PAGE_EXECUTE and suspicious content |
| `hivelist / printkey` | Registry hives and specific key values |
| `filescan` | File handles open in memory |

---


## 13.5 Network Forensics

### PCAP Analysis

Network packet captures record the complete conversation between hosts. Wireshark and Zeek analyze
PCAPs. Investigative questions: Which hosts communicated with the C2 IP? Were credentials transmitted
in the clear? What files were transferred? What DNS queries preceded the attack?

#### Network Evidence Sources

- **Firewall logs**: connection records with source/destination, port, bytes, and action.
- **Proxy logs**: full URL and user-agent for web traffic.
- **DNS logs**: all queries and responses, with timestamps.
- **NetFlow**: flow-level summaries (no payload) for high-volume environments.
- **Full packet capture**: complete payload; highest fidelity but large storage requirement.

---


## 13.6 Anti-Forensics

Anti-forensic techniques attempt to prevent, delay, or mislead forensic investigation. Common techniques:

- **Secure file deletion**: overwriting data before deletion (DoD 5220.22-M wiping) prevents carving.
- **Encryption**: encrypted volumes require the key; without it, content is inaccessible.
- **Timestomping**: modifying file timestamps to confuse timeline analysis.
- **Log deletion**: clearing Windows Event Logs, clearing bash history, or disabling logging.
- **Steganography**: hiding data in innocuous files to exfiltrate without detection.
- **Fileless malware**: executing entirely in memory via PowerShell or WMI to avoid disk artefacts.

### Countermeasures

Centralized, append-only log storage (SIEM) makes local log deletion ineffective. Immutable cloud
logging services (AWS CloudTrail with Object Lock) survive even a full EC2 instance compromise.
Memory acquisition before shutdown captures fileless malware. Monitoring for log-clearing events
(Windows Event ID 1102 for Security log cleared) provides an alert when log deletion is attempted.

---



<!--AIFOR-->

## 13.7 Artificial Intelligence in Digital Evidence Triage

The forensic process described so far assumes a human examiner can review the evidence, but modern cases bury
investigators under terabytes from disks, phones, cloud accounts, and IoT devices, which is why **AI-driven
evidence triage** has become essential. Triage is the early-stage prioritization of which artifacts deserve
deep analysis, and where traditional triage is static and rule-based, AI brings machine learning, deep
learning, and neural networks to automate **pattern recognition, anomaly detection, and artifact
classification** (as surveyed by Zamil and Khan, 2025). Natural-language processing scans emails, chat logs,
and documents to surface relevant material; image classifiers flag illicit media; and models learn to separate
the forensically interesting from the mundane far faster than manual review.

The benefits are real, but so are the limits, and a forensic examiner must understand both. AI triage helps
with data complexity, scale, and even anti-forensic countermeasures, but the recurring challenges are
**limited and biased training datasets**, **legal admissibility**, and **explainability**, an opaque model
that flags a file but cannot justify why is a liability in court. This is why the field is converging on
**explainable AI (XAI)** for forensics: a model's output must be interpretable and defensible under the
evidentiary standards (Daubert/Frye, Chapter 18) that govern expert testimony. AI accelerates the examiner; it
does not replace the chain of custody, validation, and human judgment that make evidence admissible.


## 13.8 IoT Forensics and Explainable AI

The Internet of Things multiplies both the sources of evidence and the difficulty of collecting it. **IoT
forensics** extends digital forensics to the data on smart devices, their networks, and the cloud services
behind them, and it is forensically valuable because IoT devices are now both *targets* and *witnesses*: a
breach may run through them (DDoS, ransomware, data theft), and their sensor and log data can reconstruct what
happened (Gopinath et al., 2023). The challenges are acute: devices are heterogeneous and proprietary, storage
is tiny and volatile, data lives partly in the cloud, and there are few standards, so investigators rely on
general tools, **Autopsy**, **FTK Imager**, and **Wireshark** (Chapters 3 and 8), adapted to device images and
network captures, walking the standard phases (identification, preservation, acquisition, analysis,
presentation) under chain of custody even in "data stolen" or "data deleted" scenarios.

AI is reshaping this subfield too. The **IoT-CAD** work (Mohamed et al., 2025) builds a comprehensive IoT
forensics dataset for **Cyberattack Attribution Detection**, collecting traces from Windows and Linux hosts
(memory, disk, processes, system calls, and network traffic) across realistic IoT attack scenarios, then
evaluating deep-learning attribution under both **centralized** and **federated learning** (the
privacy-preserving paradigm of Chapter 17) with **explainable-AI** techniques and network forensics. The
lesson, consistent with the triage discussion above, is that attribution and triage increasingly depend on
high-quality datasets and on explanations an investigator can defend, which is why XAI is becoming a forensic
requirement rather than a nicety.


## 13.9 AI-Driven Cybercrime Analytics and Attribution

Beyond triage and IoT, AI is changing how investigators detect and attribute cybercrime at scale, the meeting
point of forensics (this chapter), detection (Chapter 12), and threat intelligence. As a concrete example,
Djenna and colleagues (2023) combine an **unsupervised Long Short-Term Memory (LSTM)** (to model normal sequential behavior) with a
**supervised CNN** (to classify) for early detection of **botnet** attacks, evaluated on the standard
**CTU-13** and **IoT-23** datasets and reporting very high accuracy (over 98.7%) at a low false-positive rate
(around 0.04%). The forensic relevance is twofold: such models triage which hosts in a captured dataset are
compromised, and they enrich **cyber threat intelligence** by recognizing emerging botnet families, feeding
both the investigation and the defense.

```{admonition} Worked Example: AI-assisted triage on a seized dataset
:class: note
Suppose an investigator seizes a 4 TB image plus a week of network captures from a small IoT-enabled office.
A defensible AI-assisted workflow is: (1) hash and image the evidence (preserve originals, chain of custody);
(2) run NLP keyword and entity models over documents/email to rank likely-relevant items; (3) apply an
LSTM/CNN flow classifier (as above) to the captures to flag hosts showing botnet/C2 behavior; (4) for each
AI-flagged item, record the model's **explanation** (which features drove the score) so it can be defended;
(5) hand the prioritized shortlist to a human examiner for full manual analysis and reporting. The AI changes
*the order and speed* of review, not the requirement that conclusions rest on validated, explainable, manually
confirmed evidence.
```

```{admonition} Knowledge Check
:class: tip
1. Why is explainability (XAI) becoming a requirement, not a luxury, in AI-assisted forensics?
2. What makes IoT forensics harder than traditional disk forensics, and name two tools used in practice.
3. In the LSTM+CNN botnet example, what is each component responsible for?

*Answers:* (1) Forensic conclusions must be admissible and defensible under expert-evidence standards
(Chapter 18); an unexplained model output cannot be justified in court, so its reasoning must be interpretable.
(2) IoT devices are heterogeneous and proprietary with tiny, volatile storage and cloud-resident data and few
standards; practitioners use Autopsy, FTK Imager, and Wireshark. (3) The unsupervised LSTM models normal
sequential behavior to spot deviations, and the supervised CNN classifies traffic as botnet or benign.
```


## 13.10 Legal Admissibility and Reporting

Forensic work exists to support legal or organizational decisions, so admissibility is not an afterthought but
a design constraint. In U.S. courts, expert testimony and scientific methods are judged under the **Daubert**
standard (is the technique testable, peer-reviewed, with a known error rate, and generally accepted?) and the
**Federal Rules of Evidence**, while authenticity (the evidence is what it claims to be) and the
**chain of custody** are threshold requirements (these connect to the privacy-and-law material of Chapter 18).
Practically, this means using **validated, reproducible tools and methods**, documenting every step so an
opposing expert could repeat it, retaining hashes to prove integrity, and writing a clear report that separates
fact from interpretation and states the limits of the findings. The rise of AI-assisted triage (earlier in this
chapter) heightens these demands: an automated result must be explainable and independently verifiable to meet
the same bar. The examiner's credibility, like the evidence's, rests on rigor and transparency.

```{admonition} Knowledge Check
:class: tip
1. State the order of volatility and why it dictates collection order.
2. What is the role of a write blocker and a hash in preserving evidence integrity?
3. Why does the Daubert standard matter for both traditional and AI-assisted forensic methods?

*Answers:* (1) Collect most-volatile first, registers/cache, RAM, network state, disk, backups, because
volatile data is lost on power-off and cannot be recovered later. (2) A write blocker prevents any change to the
original media during acquisition; hashing source and image (and re-hashing later) proves the copy is identical
and unaltered. (3) Daubert requires methods to be testable, peer-reviewed, with known error rates, and
accepted; AI methods must additionally be explainable and reproducible to satisfy it, or their output is
inadmissible.
```


## 13.11 Cloud Forensics

The methods above assume an investigator can seize a disk or capture memory from a physical machine. In cloud environments that assumption breaks down: the hardware belongs to the provider, instances are ephemeral, and evidence is spread across services. Cloud forensics adapts the same principles to this model, and it depends heavily on the provider's logging and on snapshot capabilities.

Across the major providers the building blocks are similar even though the names differ. Acquisition usually means taking a snapshot of a virtual disk and exporting it to a secure, access-controlled location for analysis, and capturing volatile state before an instance is terminated. The richest evidence is often in the management and audit logs: AWS CloudTrail records control-plane API calls, Azure provides Activity Logs and Microsoft Entra sign-in logs, and Google Cloud provides Cloud Audit Logs. These show who did what, when, and from where, which is central to reconstructing an intrusion in an environment where the attacker's actions are often API calls rather than commands on a host. Practical challenges include the shared-responsibility model (the provider controls layers the investigator cannot reach), multi-tenancy, data residency across jurisdictions, and the short lifespan of resources, which makes pre-incident logging configuration and well-practiced acquisition automation essential. The legal and chain-of-custody requirements discussed earlier still apply, and provider cooperation may be needed to obtain certain records.

## 13.12 Disk Imaging and File Carving in Practice

Sections 13.1 and 13.2 established the principles; this section is the hands-on core that a digital-forensics
course spends weeks on. The first rule of disk forensics is to work on a verified copy, never the original.
A **forensic image** is a bit-for-bit duplicate of the storage medium, taken through a hardware or software
*write blocker* so the acquisition cannot alter the source, and validated by a cryptographic hash computed
before and after. On Linux the canonical tools are `dd` and its forensic derivatives `dc3dd` and `dcfldd`,
which add hashing and progress; the `.E01` (EnCase Expert Witness) format adds compression, metadata, and
per-block checksums and is produced by `ewfacquire` or FTK Imager.

```text
# Acquire a raw image with hashing (source /dev/sdb behind a write blocker)
sudo dc3dd if=/dev/sdb of=case001.dd hash=sha256 log=case001.log

# Or the EnCase E01 format with case metadata
sudo ewfacquire -t case001 -c fast -d sha256 /dev/sdb

# Verify the image matches the source
sha256sum case001.dd
```

Once imaged, analysis proceeds on the copy. The **Sleuth Kit** (TSK) command-line suite and its Autopsy GUI
parse file systems directly from an image: `mmls` prints the partition table, `fls` lists files including
deleted entries, `icat` extracts a file by inode, and `tsk_recover` bulk-extracts allocated and unallocated
content. Deleted files whose metadata still exists are recoverable with `icat`; when the metadata is gone but
the data blocks survive in unallocated space, the technique is **file carving**, which reconstructs files from
their content alone by recognizing known headers and footers (magic numbers), independent of any file system.
The idea is simple enough to implement, which is the point of the lab below.

In [ ]:
# Minimal JPEG carver: recover JPEG images from a raw disk image by magic bytes.
# JPEG starts with FF D8 FF and ends with FF D9. Educational; real carvers (scalpel,
# foremost, photorec) add many formats, fragmentation handling, and validation.
def carve_jpegs(raw: bytes):
    SOI, EOI = b"\xff\xd8\xff", b"\xff\xd9"
    out, i = [], 0
    while True:
        start = raw.find(SOI, i)
        if start == -1:
            break
        end = raw.find(EOI, start + 3)
        if end == -1:
            break
        out.append(raw[start:end + 2])   # include the 2-byte EOI marker
        i = end + 2
    return out

# Demo on a synthetic "disk" holding two JPEGs surrounded by junk.
fake = b"\x00"*16 + b"\xff\xd8\xff\xe0IMG1\xff\xd9" + b"GARBAGE" + b"\xff\xd8\xff\xe1IMG2\xff\xd9" + b"\x00"*8
found = carve_jpegs(fake)
print(f"carved {len(found)} JPEG(s)")
for n, img in enumerate(found):
    print(f"  image {n}: {len(img)} bytes, header {img[:4].hex()}")


The order in which evidence is collected matters, because some sources vanish faster than others. RFC 3227
codifies the *order of volatility*: capture the most ephemeral state first.

```{mermaid}
flowchart TB
    A[1. CPU registers and cache] --> B[2. RAM: processes, network connections, keys]
    B --> C[3. Disk, swap, and hibernation file]
    C --> D[4. Remote logs and monitoring data]
    D --> E[5. Physical configuration and topology]
    E --> F[6. Archival media and backups]
```

## 13.13 Memory Forensics with Volatility

Volatile memory holds what the disk never sees: running and hidden processes, network connections, injected
code, decrypted data, and encryption keys. Memory is acquired with tools such as WinPmem, LiME (Linux), or
AVML, producing a raw dump that is then analyzed offline with the open-source **Volatility 3** framework.
Volatility parses operating-system structures out of the raw bytes, so a typical investigation walks a
sequence of plugins from inventory to detail.

```text
# Identify processes, including ones hidden from the task list
vol -f mem.raw windows.pslist          # processes from the active list
vol -f mem.raw windows.psscan          # processes carved from pool tags (finds hidden/terminated)
vol -f mem.raw windows.pstree          # parent-child tree (spot an office app spawning cmd.exe)

# Network and command-line context
vol -f mem.raw windows.netscan         # sockets and connections
vol -f mem.raw windows.cmdline         # command line of each process

# Injected and malicious code
vol -f mem.raw windows.malfind         # private, executable memory with no backing file (injection)
vol -f mem.raw windows.dlllist         # loaded DLLs per process
vol -f mem.raw windows.handles         # open handles (files, keys, mutexes)

# Extract artifacts
vol -f mem.raw windows.dumpfiles       # recover cached files from memory
```

The analytic pattern is comparison: a process that appears in `psscan` but not `pslist` has been unlinked to
hide it; a browser or Office process that is the parent of `cmd.exe` or `powershell.exe` is the classic
signature of a macro or exploit; and `malfind` regions of memory that are private, executable, and unbacked by
a file on disk are where injected shellcode and hollowed processes live (Chapter 15). Because malware often
runs only in memory to avoid leaving disk artifacts (the fileless and living-off-the-land techniques of
Chapter 15), memory forensics is frequently the only place the truth is recorded.

## 13.14 Timeline Analysis

Individual artifacts answer *what*; a timeline answers *what happened, in what order*. Every file carries
**MACB** timestamps, meaning Modified, Accessed, Changed (metadata change), and Born (creation), and combining
these across the file system with log and registry timestamps reconstructs the sequence of an intrusion. The
Sleuth Kit builds a file-system timeline with `fls -m` piped into `mactime`; the **plaso** engine
(`log2timeline.py` and `psort.py`) goes further, extracting timestamps from hundreds of artifact types (event
logs, browser history, registry, prefetch) into a single normalized *super-timeline*.

```text
# File-system timeline (bodyfile -> human-readable, sorted)
fls -r -m C: -o 2048 case001.dd > bodyfile
mactime -b bodyfile -d 2026-01-01..2026-02-01 > timeline.csv

# Super-timeline across all artifact parsers
log2timeline.py --storage-file case.plaso case001.dd
psort.py -o l2tcsv -w supertimeline.csv case.plaso
```

The forensic skill is *temporal correlation*: pinpoint the initial-access moment (a phishing attachment
opened, a download), then read outward to see credential theft, lateral movement, and staging. Beware the
pitfalls: timestamps can be altered (timestomping, Section 13.6), clocks drift across systems, and time zones
must be normalized to UTC before correlation, or an analyst will manufacture a false sequence.

## 13.15 Windows Forensic Artifacts

Because most endpoints run Windows, an investigator must know where the operating system records activity,
often without the user's knowledge. These artifacts are the bread and butter of a forensics course.

| Artifact | What it proves | Location |
|---|---|---|
| Registry hives (SYSTEM, SOFTWARE, NTUSER.DAT) | Configuration, autoruns, recently used files, USB history | `C:\Windows\System32\config` and each user profile |
| Prefetch | That a program executed, when, and how many times | `C:\Windows\Prefetch\*.pf` |
| Windows Event Logs | Logons (4624/4625), process creation (4688), service installs, PowerShell | `C:\Windows\System32\winevt\Logs\*.evtx` |
| ShimCache and AmCache | Evidence of program execution even if the file is gone | Registry / `Amcache.hve` |
| LNK files and Jump Lists | Files a user opened, including from removed media | Recent items, `AutomaticDestinations` |
| Browser history, cache, cookies | Sites visited, downloads, searches | Per-browser profile directories |
| `$MFT`, `$LogFile`, `$UsnJrnl` (NTFS) | File creation, deletion, and rename history | NTFS metadata files |

Tools such as RegRipper (registry), PECmd and the Eric Zimmerman suite (prefetch, LNK, ShimCache), and
`wevtutil` or Chainsaw and Hayabusa (event logs, with Sigma rules from Chapter 12) parse these at scale.
The recurring lesson is that Windows is a prolific record-keeper: a suspect who deleted a file often left the
proof of its existence in ShimCache, its last execution in Prefetch, and the USB stick it went to in the
registry.

## 13.16 File System Internals for Investigators

Recovery and timeline work depend on knowing how the file system stores data, because deletion rarely erases
anything. The three dominant systems each keep records an investigator exploits.

**NTFS (Windows)** is organized around the *Master File Table* (`$MFT`), a record for every file and directory
holding its name, timestamps, size, and either the data itself (for small files, *resident* data) or pointers
to the clusters that hold it (*non-resident*). Deleting a file marks its `$MFT` record and clusters as free but
leaves both intact until reused, which is why deleted files are so often recoverable. NTFS also journals
metadata changes in `$LogFile` and records file-system events in `$UsnJrnl`, giving a second, independent
record of creation, deletion, and rename that survives even when the files are gone. *Alternate data streams*
let data hide behind a file's normal contents (`report.txt:hidden.exe`), a classic anti-forensic trick.

**ext4 (Linux)** stores metadata in *inodes* (owner, permissions, timestamps including a deletion time, and
block pointers) and records changes in a journal. Deletion zeroes some block pointers, making full recovery
harder than on NTFS, but the journal and unallocated blocks still yield evidence.

**APFS (macOS and iOS)** uses copy-on-write, so modifying a file writes new blocks rather than overwriting old
ones, and it integrates snapshots and native encryption. The copy-on-write design means prior versions of data
frequently persist, while pervasive encryption means acquisition increasingly depends on obtaining keys rather
than reading raw blocks.

The unifying investigator's lesson is that *deletion is not destruction*: a file removed by the user typically
survives in unallocated clusters, in journal records, and in metadata caches (Section 13.15) until those areas
are overwritten, which is why secure deletion (Section 17.9) and, on SSDs, TRIM (Chapter 17) are the real
adversaries of recovery, not the delete key.

## 13.17 Mobile Device Forensics

Phones are now the richest evidence source in most investigations, holding location history, messages, app
data, photos, and health and payment records. Mobile forensics differs sharply from disk forensics because the
storage is soldered in, the operating system is locked down, and everything is encrypted by default.

Acquisition proceeds at whatever level the device permits, from most to least complete: a *physical*
image (a full flash dump, rarely possible on modern encrypted devices), a *file-system* extraction (files and
databases, if the device can be unlocked), and a *logical* extraction (what the backup and sync APIs expose).
Much app data lives in **SQLite** databases, so an examiner routinely queries `sms.db`, call logs, and
messaging-app stores directly, and must recover deleted rows from SQLite's freelist and write-ahead log, the
mobile analog of file carving. The defining challenge is encryption: iOS and Android encrypt storage tied to
the passcode, so without the passcode or a vendor or forensic-tool bypass, the data is ciphertext. This is the
technical substance behind the recurring legal fights over compelled device unlocking (Chapter 18). Commercial
suites such as Cellebrite and Magnet AXIOM automate acquisition and parsing, but the examiner still validates
what the tool reports against the raw databases.

## 13.18 Email, Log, and Cloud-Account Forensics

Not all evidence sits on a device. Three server-side sources are central to modern cases.

**Email forensics** reads the full headers, which record the path a message took through mail servers
(`Received` lines, bottom to top), the true origin, and the authentication results (SPF, DKIM, DMARC, Chapter
3) that reveal spoofing. Header analysis is what distinguishes a genuine executive request from the
business-email-compromise fraud of Section 14.12.

**Log forensics** reconstructs activity from web-server, authentication, firewall, and application logs,
correlating them into the timeline of Section 13.14. The investigator must account for log rotation, gaps, and
the possibility that an attacker cleared logs, itself an indicator (event ID 1102 records log clearing).

**Cloud-account forensics** has become unavoidable as data moves to SaaS. The evidence is the provider's audit
trail: the AWS CloudTrail, Microsoft 365 Unified Audit Log, and Google Workspace logs of Section 13.11, which
record who accessed what, when, and from where. Acquisition here is an API and legal exercise (obtaining the
records, often via the provider) rather than a disk-imaging one, and pre-incident logging configuration
determines whether the evidence exists at all.

## 13.19 The Forensic Report and Expert Testimony

An investigation that cannot be explained and defended has no value. The **forensic report** documents the
scope and authorization, the tools and their versions, the evidence and its hashes, the exact steps taken (so
another examiner could reproduce them), the findings, and the conclusions, separating fact from
interpretation. Because findings may end up in court, the examiner may testify as an expert, and the evidence
must satisfy admissibility standards such as the United States *Daubert* standard, which asks whether the
method is testable, peer-reviewed, has a known error rate, and is generally accepted, and the Federal Rules of
Evidence (Section 13.10). The two forces that most often destroy a case are a broken chain of custody (Section
13.1) and analysis performed on an original rather than a verified image, both of which are avoidable by
discipline rather than skill.

### Exercises

1. A user emptied the Recycle Bin. Name three places on an NTFS volume where evidence of the deleted file may
   still exist.
2. Why is a physical acquisition often impossible on a modern smartphone, and what does the examiner obtain
   instead?
3. Reading email headers, in which direction do you read the `Received` lines to trace a message to its
   origin, and which fields reveal spoofing?
4. An investigation centers on data in a SaaS application. How does acquisition differ from imaging a laptop,
   and what determines whether the evidence exists?
5. State the two procedural failures most likely to render digital evidence inadmissible.

### Answer Key

1. The `$MFT` record (until reused), the `$LogFile` and `$UsnJrnl` journals, and the unallocated clusters that
   still hold the file's data.
2. Storage is encrypted with a key tied to the passcode and the flash is not directly readable, so without the
   passcode a full physical image is infeasible; the examiner obtains a file-system or logical extraction when
   the device can be unlocked.
3. Read `Received` lines from the bottom (earliest) upward to trace origin; SPF, DKIM, and DMARC results,
   together with mismatches between the envelope and header sender, reveal spoofing.
4. It relies on the provider's audit logs obtained through APIs or legal process rather than disk imaging, and
   whether the evidence exists depends on whether sufficient logging was configured before the incident.
5. A broken chain of custody and performing analysis on the original evidence instead of a verified,
   hash-validated image.

## 13.20 Acquisition in Practice: Live versus Dead, and Verification

The choice of *how* to acquire evidence is itself a forensic decision with consequences. **Dead (static)
acquisition** powers the system off and images the storage through a write blocker: it is the most defensible
approach because the source cannot change, but it destroys everything in memory, including running malware,
open network connections, and decryption keys. **Live acquisition** collects from a running system, capturing
volatile memory (Section 13.13) and the state of an encrypted volume while it is unlocked, at the cost of
altering the system slightly by the act of collecting. The modern default reflects the order of volatility
(Section 13.12): capture memory live first, then image the disk, and increasingly, because full-disk
encryption is now standard, live acquisition is the only way to obtain readable data at all, since a
powered-off encrypted disk is ciphertext.

Two disciplines make any acquisition admissible. A **write blocker**, hardware or software, sits between the
examiner and the source and physically prevents writes, so the mere act of examining cannot alter the
evidence. And **hash verification** computes a cryptographic digest (SHA-256) of the source and of the image;
if they match, the image is provably a faithful copy, and re-hashing later proves it has not changed since.
These two steps, plus the chain of custody of Section 13.1, are what separate evidence from an anecdote.

## 13.21 Anti-Forensics in Depth

Section 13.6 introduced anti-forensics; a forensics course treats it at length because recognizing it is half
the job. The techniques fall into four families.

*Data destruction* securely wipes evidence: overwriting files or free space, or, on self-encrypting and
solid-state drives, destroying the key so all data becomes unrecoverable instantly (the crypto-erase of
Chapter 17). *Data hiding* conceals rather than destroys: steganography embeds data inside images or media
(Chapter 2), NTFS alternate data streams (Section 13.16) attach hidden content to ordinary files, and slack
space and unallocated regions hold data the file system does not index. *Trail obfuscation* attacks the
timeline: **timestomping** alters a file's MACB timestamps to make a malicious file look old and legitimate,
and log clearing removes the record of activity (itself detectable, since Windows logs event 1102 when the
security log is cleared). *Counter-analysis* attacks the investigator's tools: malware detects virtual
machines and analysis sandboxes (Section 15.10) and behaves benignly, or corrupts file-system structures to
crash parsers.

The investigator's response is redundancy and cross-checking. Timestomping is caught because the tool that
sets the standard timestamps (`$STANDARD_INFORMATION` in NTFS) often cannot alter the parallel
`$FILE_NAME` timestamps, so a mismatch between the two, or a creation time earlier than the containing
volume's, betrays tampering. Deleted logs leave gaps and clearing events. Wiped files leave carving-recoverable
fragments if the wipe was incomplete. The lab below shows the timestamp cross-check that flags a timestomped
file.

In [ ]:
# Detect likely timestomping by comparing NTFS $STANDARD_INFORMATION (SI) and
# $FILE_NAME (FN) timestamps. Many timestomping tools change SI but not FN, and a
# creation time before the volume was formatted is impossible. Educational model.
from datetime import datetime

VOLUME_FORMAT = datetime(2025, 1, 1)   # earliest legitimate time on this volume

# Each record: (name, SI_created, FN_created) parsed from the $MFT
records = [
    ("report.docx",  datetime(2026, 3, 2, 9, 15), datetime(2026, 3, 2, 9, 15)),
    ("svchost.exe",  datetime(2019, 7, 4, 0, 0),  datetime(2026, 3, 5, 2, 41)),  # SI backdated
    ("notes.txt",    datetime(2024, 6, 1, 0, 0),  datetime(2024, 6, 1, 0, 0)),   # before format
]

for name, si, fn in records:
    flags = []
    if abs((si - fn).total_seconds()) > 3600:
        flags.append("SI/FN mismatch (timestomp)")
    if si < VOLUME_FORMAT or fn < VOLUME_FORMAT:
        flags.append("created before volume format (impossible)")
    status = "; ".join(flags) if flags else "consistent"
    print(f"{name:14} SI={si:%Y-%m-%d} FN={fn:%Y-%m-%d}  -> {status}")


## 13.22 From Artifact to Narrative

The final skill is synthesis: turning scattered artifacts into a defensible account of what happened. A
completed investigation weaves the strands of this chapter together, for example, prefetch and ShimCache
(Section 13.15) show that a malicious tool executed; the `$MFT` and `$UsnJrnl` (Section 13.16) show when it was
created and deleted; memory (Section 13.13) shows the process it injected into and the address it contacted;
network logs (Section 13.5) confirm the exfiltration; and the super-timeline (Section 13.14) orders it all into
a story with times. Each artifact alone proves little; together, and cross-checked for anti-forensic
tampering, they establish initial access, actions on objectives, and impact to the standard a court or a board
requires. This is why digital forensics is taught as a discipline of method and documentation rather than a
collection of tools: the tools change, but the obligation to work from a verified copy, preserve the chain of
custody, corroborate every claim, and report reproducibly does not.

```{admonition} News in Focus: Metadata that unmasked an author
:class: note
Digital forensic metadata has repeatedly identified people who believed they were anonymous. In a widely
reported case, a suspect corresponded with investigators through a removable disk; document metadata and
deleted-file artifacts on the media tied the disk to a specific organization's computer, and from there to an
individual. The lesson for both investigators and privacy-conscious users is the same as Section 13.15's:
ordinary work leaves a dense trail of authorship, timing, and device metadata that survives casual deletion,
so anonymity requires deliberate, disciplined counter-measures, not merely pressing delete.
```

### Exercises

1. Full-disk encryption is standard on modern laptops. Explain why this makes live acquisition often
   mandatory, and what is lost if the machine is powered off first.
2. A file's `$STANDARD_INFORMATION` creation time is 2019 but its `$FILE_NAME` creation time is last week. What
   do you conclude and why is the second timestamp more trustworthy?
3. Classify each as data destruction, data hiding, trail obfuscation, or counter-analysis: steganography;
   crypto-erase; clearing the security event log; malware that detects a sandbox and sleeps.
4. Why does no single artifact establish an intrusion, and what does an investigator do instead?

### Answer Key

1. A powered-off encrypted disk is ciphertext, so readable data can only be captured while the volume is
   unlocked on a running system; powering off loses memory (processes, keys, network state) and may render the
   disk unreadable without the key.
2. The file was likely timestomped: tools commonly backdate `$STANDARD_INFORMATION` but cannot easily alter
   `$FILE_NAME`, so the recent `$FILE_NAME` time is the more reliable creation time and the mismatch signals
   tampering.
3. Steganography = data hiding; crypto-erase = data destruction; clearing the security log = trail
   obfuscation; sandbox-detecting malware = counter-analysis.
4. Any single artifact can be absent, ambiguous, or tampered with; the investigator corroborates multiple
   independent artifacts (execution, file-system, memory, network, logs) into a cross-checked timeline.

## 13.23 Case Study: Reconstructing an Intrusion

To see the whole discipline at work, follow a composite investigation of a data-theft incident from acquisition
to conclusion. The organization received a tip that internal documents were circulating externally; the goal is
to determine what was taken, how, when, and by whom, to a standard that survives scrutiny.

*Acquisition.* Because the suspect endpoint was still running, the responder captured memory first, then took
a write-blocked disk image, hashing both (Section 13.20). Every step went into the chain-of-custody log. This
order preserved the volatile evidence a power-off would have destroyed.

*Triage and the timeline.* A super-timeline (Section 13.14) built from the file system, event logs, and browser
history put the events in order. The earliest anomaly was a Prefetch and ShimCache record (Section 13.15)
showing an unfamiliar executable running three weeks earlier, shortly after the user opened an email attachment
recorded in the mail client. Initial access was a phishing document.

*Memory findings.* The memory image (Section 13.13) showed that the executable had injected into a legitimate
process (`malfind` flagged private, executable, unbacked memory) and opened a connection to an external
address, which appeared again in the firewall logs at a regular interval, the beaconing signature of
command-and-control (Chapter 12).

*File-system and artifact findings.* The `$UsnJrnl` and `$MFT` (Section 13.16) recorded the creation of a
staging archive in a temporary directory, and its later deletion, hours before the tip arrived. Carving
(Section 13.12) recovered fragments of that archive from unallocated space, and its contents matched the leaked
documents. A registry USB history entry and an LNK file (Section 13.15) showed the same files had also been
copied to a removable drive.

*Anti-forensics check.* The staging executable's `$STANDARD_INFORMATION` timestamp was years old but its
`$FILE_NAME` timestamp was recent, the timestomping signature of Section 13.21, confirming deliberate evasion
and strengthening the finding of intent.

*Conclusion.* The report (Section 13.19) established, with cross-corroborated artifacts, a chain from phishing
to code execution, to command-and-control, to staging and exfiltration by both network and USB, with evidence
of anti-forensic tampering. No single artifact proved the case; the corroborated timeline did. The lab below
reproduces the timeline's core step, extracting and ordering logon events, in miniature.

In [ ]:
# Build a mini logon timeline from parsed Windows Security events and flag the
# suspicious pattern: a logon type 3 (network) or 10 (RDP) from a new external host
# shortly after a burst of failures (credential attack succeeding).
from datetime import datetime, timedelta

# (time, event_id, user, logon_type, src_ip)   4625=failed, 4624=success
events = [
    (datetime(2026,3,5,2,10), 4625, "svc_backup", 3, "203.0.113.9"),
    (datetime(2026,3,5,2,11), 4625, "svc_backup", 3, "203.0.113.9"),
    (datetime(2026,3,5,2,12), 4625, "svc_backup", 3, "203.0.113.9"),
    (datetime(2026,3,5,2,13), 4624, "svc_backup", 3, "203.0.113.9"),   # success after failures
    (datetime(2026,3,5,9,0),  4624, "alice",      2, "127.0.0.1"),     # normal local logon
]
events.sort()
fails = 0
print(f"{'time':16} {'evt':5} {'user':10} type  src")
for t, eid, user, lt, ip in events:
    tag = ""
    if eid == 4625:
        fails += 1
    elif eid == 4624:
        if fails >= 3 and lt in (3, 10):
            tag = "  <-- SUSPICIOUS: success after brute force from external host"
        fails = 0
    print(f"{t:%Y-%m-%d %H:%M} {eid:5} {user:10} {lt:>3}  {ip}{tag}")


## 13.24 The Forensic Toolkit

A forensics course is also a survey of tools, because each artifact class has its specialists. Knowing the
landscape lets an investigator pick the right instrument and, crucially, validate one tool's output against
another. The essentials:

| Category | Open-source | Commercial |
|---|---|---|
| Disk imaging | dc3dd, dcfldd, Guymager, FTK Imager (free) | EnCase, X-Ways |
| File-system analysis | The Sleuth Kit, Autopsy | EnCase, X-Ways, FTK |
| File carving | PhotoRec, foremost, scalpel, bulk_extractor | (built into suites) |
| Memory forensics | Volatility 3, MemProcFS | (built into suites) |
| Timeline | plaso (log2timeline), mactime | Timeline Explorer |
| Windows artifacts | Eric Zimmerman tools, RegRipper | Registry Explorer, Magnet |
| Mobile | ALEAPP, iLEAPP | Cellebrite, Magnet AXIOM, Oxygen |
| Log and SIEM triage | Chainsaw, Hayabusa, Sigma | Splunk, Elastic |

Two professional habits matter more than any tool. First, *dual-tool verification*: confirm a critical finding
with a second, independent tool, because a parser bug should never become a wrongful conclusion. Second,
*validation of the tools themselves* against known data sets (such as those from NIST's Computer Forensics Tool
Testing program), so that when an examiner testifies to a method's reliability (the Daubert standard of Section
13.19), the claim is backed by evidence. The tools change every few years; the method, verified acquisition,
preserved custody, corroborated findings, dual-tool checks, and reproducible reporting, does not, which is why
this book teaches the method first.

### Exercises

1. In the case study, list the artifacts that together established exfiltration, and state why the network log
   alone would have been insufficient.
2. In the logon-timeline lab, which two conditions together make a successful logon suspicious, and what attack
   do they indicate?
3. What is dual-tool verification and why does it matter for evidence presented in court?
4. The organization also found the staged archive had been deleted. Which two file-system artifacts would still
   reveal its former existence and contents?

### Answer Key

1. The injected process and its beacon (memory), the beacon in firewall logs, the staging archive in
   `$UsnJrnl`/`$MFT` and its carved fragments, and the USB and LNK artifacts; the network log alone shows a
   connection but not what was staged, from where, or that it was also copied to USB.
2. Three or more failed logons followed by a success, from an external source using a network or RDP logon
   type; this indicates a brute-force or credential-stuffing attack that succeeded.
3. Confirming a critical finding with a second independent tool; it guards against a single tool's parser bug
   producing a wrong conclusion, supporting the reliability the Daubert standard requires.
4. The `$MFT` record and `$UsnJrnl`/`$LogFile` journal entries (its former existence and metadata), and carved
   fragments recovered from unallocated space (its contents).

## 13.25 Windows Event Log Reference for Investigators

Windows Event Logs are the single richest artifact for reconstructing user and attacker activity, and an
investigator must know the key event IDs the way a doctor knows lab values. The most useful, drawn from the
Security, System, and PowerShell operational logs, are:

| Event ID | Log | Meaning | Investigative use |
|---|---|---|---|
| 4624 | Security | Successful logon (with a *logon type*) | Track access; type 3 = network, type 10 = RDP, type 2 = interactive |
| 4625 | Security | Failed logon | Brute force and password spraying (many 4625 then a 4624) |
| 4634 / 4647 | Security | Logoff | Bound a session's duration |
| 4672 | Security | Special privileges assigned at logon | An administrative or high-privilege logon |
| 4688 | Security | A new process was created (with command line, if enabled) | Reconstruct execution and parent-child lineage |
| 4720 / 4726 | Security | User account created / deleted | Attacker creating or removing accounts |
| 4728 / 4732 | Security | Member added to a privileged group | Privilege escalation via group membership |
| 4768 / 4769 | Security | Kerberos ticket requested | Kerberoasting and golden-ticket abuse (Chapter 9) |
| 7045 | System | A service was installed | Persistence via a new service |
| 1102 | Security | The audit log was cleared | Anti-forensics; itself a strong indicator |
| 4104 | PowerShell | Script block logging | Recover the actual PowerShell commands run, even if obfuscated |

The investigative pattern is to pivot: a suspicious 4688 process-creation event names a parent process and a
user; the user's 4624 logon events show where they came from; a nearby 7045 shows persistence; and a 1102
clearing event marks an attempt to hide it all. Because attackers know these logs matter, their absence or
clearing is itself evidence. Centralizing these logs off the host (the SIEM of Chapter 12) defeats local log
clearing, which is why event forwarding is a forensic control as much as a detection one.

## 13.26 Lab: Parsing Artifacts at Scale

Investigators rarely read logs by hand; they script the extraction and let patterns surface. The lab below
parses a batch of logon events, computes per-account and per-source statistics, and flags the accounts most
likely to be compromised, the kind of triage that turns thousands of events into a short list.

In [ ]:
# Triage logon events at scale: rank (user, source) pairs by a simple risk score
# built from failure bursts, off-hours activity, and privileged logons.
from collections import defaultdict

# (hour_of_day, event_id, user, logon_type, src_ip); 4625 fail, 4624 success, 4672 priv
log = [
    (2,4625,"svc","3","203.0.113.9"), (2,4625,"svc","3","203.0.113.9"),
    (2,4625,"svc","3","203.0.113.9"), (2,4624,"svc","3","203.0.113.9"), (2,4672,"svc","3","203.0.113.9"),
    (9,4624,"alice","2","127.0.0.1"), (14,4624,"bob","2","10.0.0.4"),
    (3,4624,"admin","10","198.51.100.5"),
]
stat = defaultdict(lambda: {"fail":0,"success":0,"priv":0,"offhours":0})
for hr, eid, user, lt, ip in log:
    s = stat[(user, ip)]
    if eid == 4625: s["fail"] += 1
    if eid == 4624: s["success"] += 1
    if eid == 4672: s["priv"] += 1
    if hr < 6 or hr > 22: s["offhours"] += 1

def risk(s):
    return s["fail"]*2 + s["priv"]*3 + s["offhours"]*2 + (5 if s["fail"] >= 3 and s["success"] else 0)

ranked = sorted(stat.items(), key=lambda kv: risk(kv[1]), reverse=True)
print(f"{'user @ source':28} risk  detail")
for (user, ip), s in ranked:
    print(f"{user+' @ '+ip:28} {risk(s):4d}  {dict(s)}")


### Exercises

1. You see event 4625 forty times for one account, then a single 4624 with logon type 3, then a 4672. Narrate
   the attack these three event IDs together describe.
2. Why is event 1102 (audit log cleared) considered evidence rather than an absence of evidence?
3. An attacker cleared the local Security log after their intrusion. Why might the evidence still exist, and
   what control ensures it?
4. In the triage lab, why do privileged logons and off-hours activity increase an account's risk score more
   than a single failed login?

### Answer Key

1. A password-guessing attack (many 4625 failures) succeeded (the 4624 network logon from the same source) and
   the account was high-privilege or granted special privileges at logon (4672), so an attacker gained
   privileged access via brute force.
2. Clearing the audit log is a deliberate anti-forensic act recorded as its own event, so its presence proves
   someone tried to destroy evidence, which is itself a strong indicator of malicious activity.
3. If logs are forwarded to a central SIEM (Chapter 12), the events survive off-host even when the local copy
   is cleared; event forwarding is the control that ensures it.
4. A single failed login is common and low-signal, whereas privileged access and activity at unusual hours are
   comparatively rare and align with attacker behavior, so they carry more weight in prioritizing which
   accounts to investigate.

## 13.27 A Disk Forensics Walkthrough with The Sleuth Kit

Section 13.12 introduced imaging and carving; this section walks an actual filesystem examination with The
Sleuth Kit (TSK), the open-source command-line suite that underlies Autopsy, so the analysis is concrete rather
than abstract. The scenario: you hold a forensic image `disk.E01` (Section 13.12) of a compromised machine and
must recover what happened without altering the evidence. Every command below reads the image only.

**Find the partitions.** A disk image contains a partition table; `mmls` prints it so you know where each
filesystem begins (the *starting sector offset* you pass to later tools):

```text
$ mmls disk.E01
      Slot      Start        End          Length       Description
002:  000:000   0000002048   0209713151   0209711104   NTFS / exFAT (0x07)
```

**List files, including deleted ones.** `fls` walks the filesystem's metadata. With the partition offset it
lists directory entries; deleted entries are marked with `*`, and each carries its metadata address (the inode
or MFT record number):

```text
$ fls -o 2048 -r disk.E01
r/r 64-128-1:   Users/alice/Documents/report.docx
* r/r 8721-128-3: Users/alice/AppData/.../invoice.exe   <- deleted file
d/d 75-144-1:   Windows/Temp
```

**Recover a deleted file by its metadata address.** `icat` streams the content of a file given its metadata
number, even after the directory entry is gone, as long as the data blocks have not been reused:

```text
$ icat -o 2048 disk.E01 8721 > recovered_invoice.exe   # carve out the deleted payload
$ sha256sum recovered_invoice.exe                      # hash for chain of custody (Section 13.7)
```

**Inspect a file's timestamps.** `istat` dumps a single record's metadata, including the MACB timestamps
(Section 13.14) from both `$STANDARD_INFORMATION` and `$FILE_NAME`, which is exactly the pair you compare to
detect timestomping (Section 13.21):

```text
$ istat -o 2048 disk.E01 8721
...
$STANDARD_INFORMATION Times:   Created: 2026-02-03 02:14:07   (attacker-set)
$FILE_NAME Times:              Created: 2026-08-14 09:31:52   (true creation)
```

**Build a timeline.** Chaining `fls -m` and `mactime` produces the body-file and then a human-readable timeline
of filesystem activity, the raw material of the super-timeline in Section 13.14:

```text
$ fls -o 2048 -m C: -r disk.E01 > bodyfile
$ mactime -b bodyfile -d 2026-08-14 > timeline.csv
```

The value of doing this by hand once, even though Autopsy automates it, is that you learn where the evidence
lives: partitions from `mmls`, files and deletions from `fls`, content from `icat`, timestamps from `istat`, and
sequence from `mactime`. Those five verbs cover the majority of dead-disk examinations, and understanding them
is what lets you defend your findings under the Daubert standard (Section 13.19) rather than trusting a button.

### Exercises

1. Which Sleuth Kit tool gives you the partition offset you must pass to the others, and why is that offset
   necessary?
2. How can `icat` recover a file whose directory entry has been deleted, and what is the limiting condition?
3. What does comparing the `$STANDARD_INFORMATION` and `$FILE_NAME` timestamps in `istat` output help you detect?
4. Which two tools combine to produce a filesystem timeline, and what is the intermediate artifact called?

### Answer Key

1. `mmls` prints the partition table and each partition's starting sector; the other tools need that offset to
   locate the filesystem within the whole-disk image.
2. `icat` reads content by metadata (inode/MFT) address rather than by directory entry, so a deleted file is
   recoverable as long as its data blocks have not yet been overwritten (reallocated).
3. Timestomping: a mismatch, especially a `$STANDARD_INFORMATION` time earlier than the `$FILE_NAME` time,
   indicates the more easily altered `$SI` timestamps were manipulated to hide the file's true age.
4. `fls -m` produces a body file, and `mactime` converts it into a human-readable timeline.

## 13.28 Windows Registry Forensics in Depth

Section 13.15 listed Windows artifacts; the registry deserves its own treatment because it is the single richest
source of user and system activity on a Windows machine, and because the same keys serve the malware persistence
of Section 15.23. The registry is stored on disk as a set of *hive* files, which the forensic analyst extracts
from an image and parses offline with tools such as RegRipper, so live-system tampering cannot affect the result.

The principal hives and where they live:

| Hive | On-disk location | What it records |
|---|---|---|
| SYSTEM | `Windows\System32\config\SYSTEM` | Services, drivers, mounted devices, USB history, time zone |
| SOFTWARE | `Windows\System32\config\SOFTWARE` | Installed software, auto-start locations, OS version |
| SAM | `Windows\System32\config\SAM` | Local user accounts and login counts |
| NTUSER.DAT | Each user's profile root | Per-user activity: run commands, opened files, typed paths |
| USRCLASS.DAT | Per-user `AppData\...\Windows` | Shellbags: evidence of folders the user browsed |

The high-value keys an examiner checks, and what each proves:

| Artifact (key) | Investigative value |
|---|---|
| Run / RunOnce | Persistence: what launches at boot or login (also Section 15.23) |
| UserAssist (NTUSER) | GUI programs the user launched, with run counts and last-run times |
| ShimCache / AppCompatCache (SYSTEM) | Programs that were present or executed, even if since deleted |
| AmCache.hve | Executed program paths with SHA-1 hashes, a strong execution artifact |
| RecentDocs / OpenSavePidlMRU (NTUSER) | Files the user recently opened or saved |
| TypedPaths, RunMRU (NTUSER) | Paths typed in Explorer and commands typed in the Run dialog |
| USBSTOR (SYSTEM) | Serial numbers and first/last connection of USB storage devices |
| Shellbags (USRCLASS) | Folders the user navigated, including now-deleted or external ones |

Two of these are worth emphasizing because they answer the question investigations most often ask, *did this
program run?* ShimCache and AmCache both record program execution or presence independently of the file still
existing, so they can prove a deleted attacker tool ran and when. UserAssist adds the human dimension by tying
GUI program launches to a specific user account with counts and timestamps.

A caution completes the picture: registry timestamps are *last-write* times of a key, not of every value, so an
examiner reasons carefully about what a timestamp actually dates, and corroborates registry findings with the
filesystem timeline (Section 13.27) and event logs (Section 13.25). Used with that discipline, the registry
turns a dead disk into a detailed account of who ran what, when, and from which device, which is frequently the
core of an intrusion or insider case.

### Exercises

1. Why does a forensic examiner parse registry hive files offline rather than reading the registry on the live
   system?
2. Which two registry-derived artifacts help prove that a now-deleted program executed, and what does each add?
3. What does the USBSTOR key establish, and in what kind of case does it matter?
4. Why must an examiner be careful when interpreting a registry key's timestamp?

### Answer Key

1. To preserve evidentiary integrity and avoid tampering; the hives are extracted from the image and parsed with
   dedicated tools so the analysis is reproducible and does not alter the live system.
2. ShimCache/AppCompatCache records programs that were present or executed even if since deleted; AmCache adds
   executed program paths together with SHA-1 hashes, strengthening attribution of the specific binary.
3. It records USB storage devices connected to the machine, including serial numbers and connection times, which
   matters in data-theft, insider, and policy-violation cases.
4. The timestamp is the key's last-write time, not a per-value time, so it may reflect a later change to any
   value under the key; findings must be corroborated with filesystem and log evidence.

## 13.29 Artificial Intelligence in Digital Forensics

The volume of data in a modern investigation, terabytes of disk images, memory, and network captures across many
devices, has outpaced what a human examiner can review manually, so machine learning is increasingly applied to
the forensic process itself. This section situates that trend and points to the current literature, because it is
where the field is heading and a natural extension of the triage and timeline skills built in this chapter.

The clearest near-term application is *evidence triage*: using machine-learning models to rank which of thousands
of files, images, or artifacts an examiner should look at first, so scarce expert attention goes to the most
probative material. Recent surveys map how deep-learning and hybrid models are being used to sort and prioritize
evidence, and report both efficiency gains and open problems in reliability and validation (Zamil and Khan,
2025). A second application is *attribution*: correlating attack artifacts to identify the actor or campaign
behind an intrusion, an area held back mainly by the scarcity of good labeled data, which purpose-built forensic
datasets aim to address (Mohamed et al., 2025). A third is broad *analytics over cybercrime data*, where
unsupervised sequence models flag malicious behavior at scale (Djenna et al., 2023). And because so much modern
evidence originates on constrained, heterogeneous Internet-of-Things devices, a growing line of work studies IoT
forensics specifically, with an emphasis on *explainability*, so that a model's output can be understood and
defended rather than trusted blindly (Gopinath et al., 2023).

The recurring caution across this literature is the one that matters most for practice: a forensic conclusion
must be admissible and defensible (Section 13.19), so a machine-learning result that cannot be explained,
validated, and reproduced is a lead, not evidence. AI in forensics is therefore best understood as a way to
*focus* human expertise, ranking, clustering, and surfacing candidates, rather than as a replacement for the
examiner's judgment and the chain of reasoning a court requires. Used that way, it extends the triage discipline
of Section 13.26 to a scale that manual review can no longer reach.

For readers who want to go deeper, the four works cited here, a review of AI-driven evidence triage (Zamil and
Khan, 2025), an explainable IoT-forensics study (Gopinath et al., 2023), a forensic dataset for AI-based attack
attribution (Mohamed et al., 2025), and an AI-driven cybercrime-analytics approach (Djenna et al., 2023), are a
representative entry point to the current research.

### Exercises

1. Why has machine learning become attractive for digital forensics in particular?
2. What is evidence triage, and what does an ML model contribute to it?
3. Why is explainability emphasized in forensic machine learning more than in many other applications?
4. Why should an unexplained machine-learning result be treated as a lead rather than as evidence?

### Answer Key

1. The volume and variety of data in modern cases (terabytes across many devices) exceeds what examiners can
   review manually, so ML is used to prioritize and scale the work.
2. Triage is deciding which artifacts to examine first; an ML model ranks files, images, or events by likely
   relevance so expert attention is spent on the most probative material.
3. Because forensic findings must be admissible and defensible in court; an examiner has to explain, validate,
   and reproduce a result, which an unexplained (black-box) model output cannot support.
4. Because it cannot be explained, validated, and reproduced to the standard evidence requires (Section 13.19),
   so it should focus further human investigation rather than stand on its own as a conclusion.

## Chapter Summary

This chapter presented digital forensics as a disciplined, evidence-centered practice. It established the forensic principles of integrity and chain of custody, then worked through evidence acquisition and analysis across file systems, memory, and the network, followed by anti-forensics and how investigators counter it. It extended into modern methods, including artificial intelligence for digital evidence triage, IoT forensics with explainable AI, and AI-driven cybercrime analytics and attribution, and closed with legal admissibility and reporting. The throughline is that sound methodology and defensible documentation are what make technical findings hold up, in court and in incident review alike.

## Why This Matters

Digital forensics is both a technical discipline and a legal process. Evidence that is improperly
collected or handled cannot be used in court and may allow a perpetrator to avoid consequences.
For corporate investigations, improperly handled forensic evidence can also expose the organization
to civil liability. Technical staff who understand forensic principles make better first responders:
they preserve volatile evidence, avoid contaminating the scene, and document their actions in a way
that supports subsequent investigation.

---


## News in Focus: When Digital Forensic Evidence Decides a Case
Several high-profile criminal prosecutions have turned on digital forensic evidence: recovered
deleted chat logs, memory dumps containing encryption keys, and network captures proving communication
with known malicious infrastructure. Equally, cases have been dismissed or weakened because
investigators failed to maintain chain of custody, used non-forensic acquisition methods, or
could not prove that evidence had not been altered. The technical quality of forensic work directly
determines whether justice is served.

---


In [1]:
# Chapter 13 -- Hash verification, file timestamp analysis, and artefact recovery simulation
import hashlib, os, time
from datetime import datetime
from io import BytesIO
from IPython.display import display, Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Hash verification demo ────────────────────────────────────────────────────
def sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

original_image = b"FORENSIC_IMAGE_" + b"A" * 1000
img_hash = sha256(original_image)
print("=== Evidence Integrity Verification ===")
print(f"  Original image SHA-256 : {img_hash}")

verified = sha256(original_image)
print(f"  Re-verified SHA-256    : {verified}")
print(f"  Integrity check        : {'PASSED' if verified == img_hash else 'FAILED - TAMPERED!'}")

tampered = original_image[:-1] + b"B"
print(f"  Tampered image SHA-256 : {sha256(tampered)}")
print(f"  Tampered check         : {'PASSED' if sha256(tampered)==img_hash else 'FAILED - TAMPERED!'}")

# ── File timeline reconstruction ──────────────────────────────────────────────
print("\n=== NTFS Timeline Artefacts (Simulated) ===")

artefacts = [
    ("2026-05-28 08:02:11", "LOGIN",    "Administrator logged on (Event 4624)"),
    ("2026-05-28 08:14:33", "DOWNLOAD", "7za.exe downloaded to C:\\Users\\admin\\Downloads"),
    ("2026-05-28 08:15:01", "EXEC",     "7za.exe executed (Prefetch: 7za.exe-DEADBEEF.pf)"),
    ("2026-05-28 08:15:10", "ARCHIVE",  "archive.7z created in C:\\Users\\admin\\Documents"),
    ("2026-05-28 08:17:44", "NETWORK",  "HTTPS connection to 198.51.100.42:443 (62 MB transferred)"),
    ("2026-05-28 08:19:01", "DELETE",   "archive.7z deleted (MFT entry 0x3A9C marked free)"),
    ("2026-05-28 08:19:45", "LOGOUT",   "Administrator logged off (Event 4634)"),
]

for ts, etype, desc in artefacts:
    print(f"  {ts}  [{etype:<8}]  {desc}")

print("\n  Narrative: Administrator downloaded and executed an archiver, created an archive,")
print("  transferred 62 MB externally over HTTPS, then deleted the archive. Exfiltration suspected.")

# ── Memory forensics: simulated process list ──────────────────────────────────
print("\n=== Simulated Volatility pslist Output ===")
processes = [
    ("System",       4,    0,  "SYSTEM",    False),
    ("lsass.exe",    668,  568,"SYSTEM",    False),
    ("svchost.exe",  900,  568,"NETWORK",   False),
    ("explorer.exe", 2340, 1200,"ADMIN",    False),
    ("powershell.exe",3100,2340,"ADMIN",    True),
    ("cmd.exe",      3220,3100,"ADMIN",     True),
    ("7za.exe",      4100,3220,"ADMIN",     True),
]

print(f"  {'Name':<20} {'PID':>6} {'PPID':>6} {'User':<10}  Flag")
print("  " + "-"*55)
for name, pid, ppid, user, suspicious in processes:
    flag = " <-- SUSPICIOUS (spawned by Explorer)" if suspicious else ""
    print(f"  {name:<20} {pid:>6} {ppid:>6} {user:<10} {flag}")


=== Evidence Integrity Verification ===
  Original image SHA-256 : a3170f0ac498b9d58f5c46ee1f356984b9300f6f4c374918d347508c8d013efa
  Re-verified SHA-256    : a3170f0ac498b9d58f5c46ee1f356984b9300f6f4c374918d347508c8d013efa
  Integrity check        : PASSED
  Tampered image SHA-256 : 2153377eee63fb0c748430ab848937cfd5b90f9d4e1ec33a50693de961ca1988
  Tampered check         : FAILED - TAMPERED!

=== NTFS Timeline Artefacts (Simulated) ===
  2026-05-28 08:02:11  [LOGIN   ]  Administrator logged on (Event 4624)
  2026-05-28 08:14:33  [DOWNLOAD]  7za.exe downloaded to C:\Users\admin\Downloads
  2026-05-28 08:15:01  [EXEC    ]  7za.exe executed (Prefetch: 7za.exe-DEADBEEF.pf)
  2026-05-28 08:15:10  [ARCHIVE ]  archive.7z created in C:\Users\admin\Documents
  2026-05-28 08:17:44  [NETWORK ]  HTTPS connection to 198.51.100.42:443 (62 MB transferred)
  2026-05-28 08:19:01  [DELETE  ]  archive.7z deleted (MFT entry 0x3A9C marked free)
  2026-05-28 08:19:45  [LOGOUT  ]  Administrator logged off (

## Review Questions (MCQ)

**Q1.** The chain of custody primarily ensures:
A. Evidence is encrypted  B. Any person who handled evidence is documented, supporting legal admissibility  C. The forensic image is compressed  D. The hard drive is wiped after analysis

**Q2.** A write blocker is used to:
A. Prevent the suspect from accessing the system  B. Prevent any write to the source device during acquisition  C. Encrypt the forensic image  D. Block network traffic during imaging

**Q3.** A physical forensic image differs from a logical image in that it:
A. Is faster to create  B. Includes unallocated space, deleted files, and slack space  C. Only copies active files  D. Is compressed

**Q4.** NTFS stores four timestamps per file. The acronym for this set is:
A. CRUD  B. MACB (Modified, Accessed, Changed, Born)  C. ACID  D. ITAR

**Q5.** Fileless malware specifically evades which forensic technique?
A. Memory forensics  B. Network forensics  C. Disk-based file carving and artefact recovery  D. Hash verification

**Q6.** Volatility's `malfind` plugin looks for:
A. Running processes  B. Network connections  C. Memory regions marked executable with suspicious content  D. Registry hives

**Q7.** Secure file deletion (overwriting before delete) defeats:
A. Memory forensics  B. File carving from unallocated space  C. Log analysis  D. Timestamp correlation

**Q8.** Windows Event ID 1102 indicates:
A. A new process created  B. The Security event log was cleared  C. A privilege escalation  D. A failed login

**Q9.** NetFlow differs from full packet capture in that NetFlow:
A. Captures payload content  B. Records only flow-level metadata (no payload)  C. Requires a write blocker  D. Only records TCP traffic

**Q10.** Which hash algorithm is preferred for new forensic investigations?
A. MD5  B. SHA-1  C. SHA-256  D. CRC-32

*Answers: Q1 B, Q2 B, Q3 B, Q4 B, Q5 C, Q6 C, Q7 B, Q8 B, Q9 B, Q10 C.*


## Lab Assignment

**Part A -- Image acquisition**: Create a test forensic image of a USB drive or a virtual disk using `dd` or FTK Imager. Compute SHA-256 before and after acquisition. Verify that the hashes match. Document the command used and the exact hash values.

**Part B -- File carving**: Using Autopsy or PhotoRec on the image from Part A (or a provided sample image), recover at least three deleted files. Document the file type, recovered content, and whether the file was recoverable because TRIM was not applied.

**Part C -- Timeline creation**: Using Autopsy's timeline feature (or `mactime` from The Sleuth Kit), extract a sorted timeline of file system artefacts from a test image. Identify a five-minute window of intense activity and describe what a forensic investigator would infer from it.

**Part D -- Memory analysis**: Obtain a publicly available Windows memory sample (from Volatility's test images). Run `pslist`, `netscan`, and `cmdline`. Identify any process that should not be running (unusual parent-child relationship, suspicious command line, or network connection to a non-standard port).


## References

```{bibliography}
:filter: docname in docnames
```

1. Zamil, M. Z. H., and Khan, T. M. (2025). AI-Driven Digital Evidence Triage in Digital Forensics: A Comprehensive Review. IEEE ISDFS 2025.
2. Mohamed, H., Koroniotis, N., Schiliro, F., and Moustafa, N. (2025). IoT-CAD: A Comprehensive Digital Forensics Dataset for AI-based Cyberattack Attribution Detection in IoT. Ad Hoc Networks 174, 103840.
3. Gopinath, A., Kukatlapalli, P. K., Saleem K. M., S., and John, J. (2023). Explainable IoT Forensics: Investigation on Digital Evidence. IEEE InC4 2023.
4. Djenna, A., Barka, E., Benchikh, A., and Khadir, K. (2023). Unmasking Cybercrime with Artificial-Intelligence-Driven Cybersecurity Analytics. Sensors 23(14), 6302.
5. NIST SP 800-86: Guide to Integrating Forensic Techniques into Incident Response.
6. ACPO Good Practice Guide for Digital Evidence; The Sleuth Kit/Autopsy; Volatility Foundation.


```{index} Order of volatility, Chain of custody, Write blocker / forensic image / hashing, Daubert standard / Federal Rules of Evidence, AI evidence triage, IoT forensics, Explainable AI, Cyberattack attribution detection, Forensic image, Write blocker, Hash verification, Deleted file recovery, Slack space, MFT, Memory forensics, Volatility, PCAP, Artefact, Anti-forensics
```
